# Accuracy Comparison

In [1]:
# Loading libraries

import numpy as np
import pandas as pd

In [2]:
# Importing data

df = pd.read_csv('predictions_data.csv')
df.drop('Unnamed: 0', axis=1, inplace=True)
df.info()
df.sample(10, random_state = 123)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Variant   36 non-null     object 
 1   Index     36 non-null     object 
 2   Approach  36 non-null     object 
 3   Leader    36 non-null     object 
 4   Forecast  36 non-null     float64
dtypes: float64(1), object(4)
memory usage: 1.5+ KB


,Variant,Index,Approach,Leader,Forecast
6,baseline,WG,BERT,fayulu,0.589260
8,baseline,WG,BERT,tshisekedi,0.212872
13,baseline,YHRM,BERT,ramazani,0.271532
11,baseline,WG,VADER,tshisekedi,0.194274
5,baseline,TSSW,VADER,tshisekedi,0.385786
32,penalized,YHRM,BERT,tshisekedi,0.441688
23,penalized,TSSW,VADER,tshisekedi,0.432994
31,penalized,YHRM,BERT,ramazani,0.214446
12,baseline,YHRM,BERT,fayulu,0.371352
18,penalized,TSSW,BERT,fayulu,0.268622


In [3]:
# Observed vote shares from CENI data (benchmark)
observed = {'fayulu': 0.3482, 'ramazani': 0.2383, 'tshisekedi': 0.3856}

In [4]:
# Accuracy function

def accuracy_score(observed_dict, predicted_dict):
    """
    Calculate accuracy score using the relative-absolute formula.
    
    Parameters:
    observed_dict (dict): Observed vote shares {candidate: share}
    predicted_dict (dict): Predicted vote shares {candidate: share}
    
    Returns:
    float: Accuracy score between 0 and 1
    """
    K = len(observed_dict)  # Number of candidates
    total_error = 0
    
    for candidate, observed_share in observed_dict.items():
        if candidate in predicted_dict:
            predicted_share = predicted_dict[candidate]
            # Avoid division by zero
            if observed_share > 0:
                relative_error = abs(predicted_share - observed_share) / observed_share
                total_error += relative_error
    
    accuracy = 1 - (total_error / K)
    return accuracy

In [5]:
# Create accuracy results
accuracy_results = []

# Iterate through all combinations in final_df
for (variant, index, approach), group in df.groupby(['Variant', 'Index', 'Approach']):
    # Convert group to predicted dictionary
    predicted_dict = dict(zip(group['Leader'], group['Forecast']))
    
    # Calculate accuracy
    accuracy_values = accuracy_score(observed, predicted_dict)
    
    accuracy_results.append({
        'Variant': variant,
        'Index': index,
        'Approach': approach,
        'Accuracy': accuracy_values
    })

# Create accuracy dataframe
accuracy_df = pd.DataFrame(accuracy_results)

In [6]:
# Pivot to get the desired format: indices as columns, variant-approach combinations as rows
accuracy_pivot = accuracy_df.pivot_table(
    index=['Variant', 'Approach'],
    columns='Index',
    values='Accuracy'
).reset_index()

# Sort for consistent ordering
accuracy_pivot = accuracy_pivot.sort_values(['Variant', 'Approach'])

# Rename the index to combine variant and approach
accuracy_pivot['Model'] = accuracy_pivot['Variant'] + '_' + accuracy_pivot['Approach']
accuracy_pivot = accuracy_pivot[['Model', 'TSSW', 'WG', 'YHRM']]

print(f"{'='*50}\nFinal Accuracy Table:\n{'='*50}")
accuracy_pivot.round(4)

Final Accuracy Table:


Index,Model,TSSW,WG,YHRM
0,baseline_BERT,0.8251,0.5634,0.9067
1,baseline_VADER,0.7807,0.5909,0.8888
2,penalized_BERT,0.8096,0.6401,0.9140
3,penalized_VADER,0.8568,0.7148,0.9003


In [7]:
# Export to CSV
csv_filename = "accuracy_data.csv"
accuracy_pivot.to_csv(csv_filename)
print(f"\nDataFrame successfully exported to: {csv_filename}")


DataFrame successfully exported to: accuracy_data.csv
